# Lab 9: Phylogenetic Analysis

Build, visualize, and interpret phylogenetic trees from real protein sequences using Biopython and ETE3.

**Goals:**
- Understand the difference between distance-based and character-based tree-building methods.
- Compute a multiple sequence alignment and derive a distance matrix from it.
- Build UPGMA and Neighbor-Joining trees and compare their topologies.
- Evaluate branch support with bootstrap resampling.
- Interpret a phylogenetic tree in a biological context.

---
## Part 0 — Environment Setup

This time you will use your own laptop (not the Silo server), in two ways
- VS Code on your local machine -- follow the steps in this notebook (Plan B: if you cannot make it work, go with google colab)
- Using a standalone program MEGA v12, which you can download from https://megasoftware.net/. Install and run it on your local machine. Follow their tutorials.

Dataset: you can get a small collection of proteins (RTs) from the Bio2 repo, under datasets/phylogeny/allRT.faa

In [1]:
# Run once, afterwards you can comment it out
%pip install -q biopython numpy matplotlib pandas ete3 PyQt5


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Check if the required packages are installed properly

In [ ]:
from io import StringIO

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment

print("All imports successful ✓")

---
## Background

A **phylogenetic tree** depicts evolutionary relationships among organisms or genes.
Two broad approaches exist:

| Approach | Key idea | Examples |
|---|---|---|
| **Distance-based** | Convert alignment → pairwise distances → tree | UPGMA, Neighbor-Joining (NJ) |
| **Character-based** | Score each column of the alignment directly | Maximum Parsimony, Maximum Likelihood, Bayesian |

This lab focuses on distance-based methods because they are fast and illustrative.
Character-based methods are more accurate for real research and are covered in lecture.

Key vocabulary:
- **OTU** (Operational Taxonomic Unit) — the leaf nodes (sequences or taxa) of the tree.
- **Clade** — a group that includes an ancestor and all its descendants.
- **Branch length** — the amount of evolutionary change along a branch.
- **Bootstrap support** — the fraction of resampled datasets that recover the same clade (0–100).

---
## Part 1 — Distance Matrices by Hand

Before using software, let's build intuition for distance matrices with a tiny example.

### 1a. Compute pairwise Hamming distances

The **Hamming distance** between two equal-length sequences is the number of positions where they differ.
For aligned sequences of length *L*, the fractional distance is `d = mismatches / L`.

In [ ]:
seqs = {
    "Human":  "ACGTTAGCATGCTAGCATCG",
    "Chimp":  "ACGTTAGCATGCTAGCATCA",
    "Gorilla":"ACGTTAGCATGCAAGCATCG",
    "Mouse":  "ACGCTAGCATGCTAGTATCG",
    "Zebrafish":"TCGCTAGAATGCTAGTATCC",
}

names = list(seqs.keys())
L = len(next(iter(seqs.values())))

# Build distance matrix
dist = pd.DataFrame(0.0, index=names, columns=names)
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i < j:
            d = sum(x != y for x, y in zip(seqs[a], seqs[b])) / L
            dist.loc[a, b] = d
            dist.loc[b, a] = d

print(f"Sequence length: {L} bp")
display(dist.round(3))

> **Questions 1a:1–3**
>
> 1a.1. Which pair of sequences is most similar? Does that match your biological expectation?
>
> 1a.2. Which pair is most distant? Is Zebrafish equally distant from all mammals?
>
> 1a.3. Is the distance matrix symmetric? Why must it be?

**Your answers:**

1a.1.

1a.2.

1a.3.

### 1b. Visualise the distance matrix as a heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(dist.values, cmap="YlOrRd", vmin=0, vmax=0.3)
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=35, ha="right")
ax.set_yticklabels(names)
plt.colorbar(im, ax=ax, label="Fractional distance")
ax.set_title("Pairwise Hamming Distance Heatmap")
plt.tight_layout()
plt.show()

> **Question 1b**
>
> 1b.1. Do you see a block structure in the heatmap? What does that tell you about the groupings?

**Your answer:**

1b.1.

---
## Part 2 — UPGMA and Neighbor-Joining Trees

We will now build trees from a real multiple sequence alignment.
The dataset is the **cytochrome c** protein — a core component of the mitochondrial electron transport chain present in virtually all eukaryotes.

### 2a. Load and inspect the alignment

In [ ]:
# Pre-aligned cytochrome c sequences (104 aa, Clustal format)
# Sources: UniProt P99999 (Human), P00058 (Tuna), P00164 (Yeast), P00166 (Maize), P00174 (Candida)
cytc_fasta = """\
>Human
MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE
>Chimpanzee
MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE
>Horse
MGDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKSERVDLIAYLKKATNE
>Tuna
MGDVAKGKKTFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAAGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKSERVDLIAYLKKATNE
>Drosophila
MGVPAGDVEKGKKLFVQRCAQCHTVEAGGKHKVGPNLHGLIGRKTGQAEGFSYTDANKNKGITWGEDTLMEYLENPKKYIPGTKMIFAGIKKKDERVDLIAYLKDATSK
>Yeast
MGSAKAGDVEKGKKLFVQRCAQCHTVEKGGKHKTGPALHGLFGRKTGQAEGFSYTDANKSKGIVWNNDTLMEYLENPKKYIPGTKMIFAGIKKKGERADLIAYLKKATSS
>Maize
MGDAAKGEKEFNKSKACHTVDKGAKHKTGPNLHGLFGRQSGTTPGYSYSAANKNMAVIWEEHTLYDYLLNPKKYIPGTKMVFPGLKKPQERADLIAYLKESTK-
"""

records = list(SeqIO.parse(StringIO(cytc_fasta), "fasta"))

summary = pd.DataFrame([
    {"ID": r.id, "Length": len(r.seq), "First 20 aa": str(r.seq[:20])}
    for r in records
])
display(summary)

> **Questions 2a:1–2**
>
> 2a.1. Why are the sequences different lengths? What does a gap character (`-`) represent?
>
> 2a.2. Cytochrome c is described as "highly conserved". Do the sequences look similar at a glance?

**Your answers:**

2a.1.

2a.2.

### 2b. Build a distance matrix from the alignment

In [ ]:
# Pad sequences to the same length (fill with gaps)
max_len = max(len(r.seq) for r in records)
padded  = [SeqRecord(Seq(str(r.seq).ljust(max_len, "-")), id=r.id) for r in records]
alignment = MultipleSeqAlignment(padded)

calculator = DistanceCalculator("identity")   # fractional identity
dm = calculator.get_distance(alignment)

# Pretty-print as a DataFrame
taxa = dm.names
df_dm = pd.DataFrame(
    [[round(dm[i, j], 4) for j in taxa] for i in taxa],
    index=taxa, columns=taxa
)
display(df_dm)

> **Questions 2b:1–2**
>
> 2b.1. Which two taxa have the smallest distance? Are they the ones you expected?
>
> 2b.2. Which taxon is most distant from Human? Does that match evolutionary expectations?

**Your answers:**

2b.1.

2b.2.

### 2c. Build UPGMA and Neighbor-Joining trees

In [ ]:
constructor = DistanceTreeConstructor(calculator)

upgma_tree = constructor.upgma(dm)
nj_tree    = constructor.nj(dm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tree, method in zip(axes, [upgma_tree, nj_tree], ["UPGMA", "Neighbor-Joining"]):
    Phylo.draw(tree, axes=ax, do_show=False)
    ax.set_title(method, fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print("\n=== UPGMA (Newick) ===")
Phylo.write(upgma_tree, sys.stdout if False else StringIO(), "newick")
from io import StringIO as _SIO
buf = _SIO()
Phylo.write(upgma_tree, buf, "newick")
print(buf.getvalue())

print("\n=== Neighbor-Joining (Newick) ===")
buf2 = _SIO()
Phylo.write(nj_tree, buf2, "newick")
print(buf2.getvalue())

> **Questions 2c:1–3**
>
> 2c.1. Do the UPGMA and NJ trees agree on the grouping of Human, Chimpanzee, and Horse?
>
> 2c.2. UPGMA assumes a **molecular clock** (all lineages evolve at the same rate). Does the topology look consistent with that assumption?
>
> 2c.3. Where does Yeast fall in the tree relative to the animals? Is that biologically reasonable?

**Your answers:**

2c.1.

2c.2.

2c.3.

### 2d. Try a different distance model

The `identity` model simply counts mismatches. For proteins, substitution matrices (e.g., BLOSUM62) capture evolutionary likelihood better.
Biopython's `DistanceCalculator` supports several models; try `"blosum62"` below.

In [31]:
# your work here — rebuild the NJ tree using the "blosum62" model and plot it


> **Question 2d**
>
> 2d.1. Does the tree topology change when you switch to BLOSUM62? Which model do you trust more, and why?

**Your answer:**

2d.1.

---
## Part 3 — Bootstrap Support

A tree topology can change depending on which sites of the alignment are included.
**Bootstrap resampling** randomly resamples alignment columns (with replacement) and rebuilds the tree many times.
The fraction of resampled trees that recover a given clade is its **bootstrap support value** (0–100).

Values ≥ 70 are generally considered well-supported.

### 3a. Manual bootstrap (NJ)

In [ ]:
from Bio import AlignIO
from Bio.Phylo.Consensus import bootstrap_trees
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor


# Configure the calculator and tree constructor
calculator = DistanceCalculator("blosum62")
constructor = DistanceTreeConstructor(calculator)

# Generate 100 bootstrap replicate trees
# Note: bootstrap_trees returns a generator, so we cast it to a list
replicate_trees = list(bootstrap_trees(alignment, 100, constructor))

# Build a consensus tree from the bootstrap replicates
from Bio.Phylo.Consensus import majority_consensus, strict_consensus

# Create a majority-rule consensus tree (cutoff defaults to 0.5/50%)
consensus_tree = majority_consensus(replicate_trees, cutoff=0.5)

# Show the tree
fig, ax = plt.subplots(figsize=(7, 5))
Phylo.draw(consensus_tree, axes=ax, do_show=False)

> **Questions 3a:1–2**
>
> 3a.1. Which clades have high bootstrap support (≥ 70)? Which are poorly supported?
>
> 3a.2. A bootstrap value of 50 means a clade appeared in exactly half the replicates. Should you trust such a split?

**Your answers:**

3a.1.

3a.2.

### 3b. Effect of replicate count on consensus

Rerun the bootstrap with `n_replicates = 500` and compare. Is the topology more stable?

In [33]:
# your work here


> **Question 3b**
>
> 3b.1. Did the support values change noticeably with 500 replicates? What does this suggest about using 100 vs 1000 replicates in practice?

**Your answer:**

3b.1.

---
## Part 4 — Your Own Phylogeny

### 4a. Retrieve sequences from NCBI or other places

Choose **one** of the following gene families and fetch at least **six** protein sequences from diverse taxa.

| Gene family | Why it's interesting |
|---|---|
| **16S rRNA** (prokaryotes) | Universal phylogenetic marker; standard for bacterial taxonomy |
| **rbcL** (plants) | Photosynthesis enzyme; used for plant barcoding |
| **COX1** | Mitochondrial gene; used for animal DNA barcoding |
| **RecA / RAD51** | DNA repair; highly conserved across all domains of life |



### 4. Get your own sequences, Build and plot your phylogenetic tree

Pad / align your sequences, compute a distance matrix, and build an NJ tree with bootstrap support.
Use any approach that works for you: copy and adapt Part 2–3, or write your own code.

In [34]:
# your work here


> **Questions 4:1–4**
>
> 4.1. Which gene family did you choose, and why?
>
> 4.2. Does your tree topology match the known evolutionary relationships for those taxa?
>
> 4.3. Are there any surprising groupings? What might explain them (e.g., gene duplication, horizontal gene transfer, convergent evolution)?
>
> 4.4. Which clades have the strongest bootstrap support, and which are weakly supported?

**Your answers:**

4.1.

4.2.

4.3.

4.4.

---
## Part 5 — Tree Comparison and Interpretation

### 5a. Robinson–Foulds distance

The **Robinson–Foulds (RF) distance** counts the number of bipartitions (splits) present in one tree but not the other.
A distance of 0 means the topologies are identical.

In [ ]:
def get_bipartitions(tree):
    """Return frozenset of frozensets representing each internal split."""
    all_taxa = frozenset(c.name for c in tree.get_terminals())
    splits = set()
    for clade in tree.get_nonterminals():
        tips = frozenset(c.name for c in clade.get_terminals())
        if tips != all_taxa:               # skip the root split
            splits.add(frozenset([tips, all_taxa - tips]))
    return splits

upgma_splits = get_bipartitions(upgma_tree)
nj_splits    = get_bipartitions(nj_tree)

sym_diff = upgma_splits.symmetric_difference(nj_splits)
rf = len(sym_diff)

print(f"UPGMA splits : {len(upgma_splits)}")
print(f"NJ splits    : {len(nj_splits)}")
print(f"RF distance  : {rf}")
print()
if rf == 0:
    print("Topologies are identical.")
else:
    print("Splits only in UPGMA:")
    for s in upgma_splits - nj_splits:
        print(" ", set(list(s)[0]))
    print("Splits only in NJ:")
    for s in nj_splits - upgma_splits:
        print(" ", set(list(s)[0]))

> **Questions 5a:1–2**
>
> 5a.1. Are the UPGMA and NJ topologies identical for the cytochrome c dataset? If not, which splits differ?
>
> 5a.2. A larger RF distance means the trees disagree more. When would you expect UPGMA and NJ to diverge most?

**Your answers:**

5a.1.

5a.2.

### 5b. Decision guide — which method should you use?

| Situation | Recommended method |
|---|---|
| Large dataset, fast exploration | Neighbor-Joining |
| Sequences evolving at similar rates | UPGMA |
| High accuracy for publication | Maximum Likelihood (IQ-TREE, RAxML) |
| Uncertainty quantification needed | Bayesian (MrBayes, BEAST) |
| Morphological / discrete characters | Maximum Parsimony |

> **Question 5b**
>
> 5b.1. For a study comparing 200 whole-genome gene trees from bacteria, which method would you choose and why?

**Your answer:**

5b.1.